# PubMed LLM Maintenance Runner

Use this notebook for routine lab maintenance. Run **Setup**, edit **Settings**, then run **Main Maintenance Run**.

The main run can handle:

- new gene requests from the website queue
- failed/error queue rows after a fix
- interrupted `processing` rows
- existing genes that need refresh based on `last_run_at`

Do not paste real passwords, API tokens, or service-account JSON into this notebook. Use Colab Secrets.

## 1. Setup

Run once at the start of a Colab session.

In [ ]:
from google.colab import drive, userdata
import os, subprocess, shlex

drive.mount('/content/drive')
%cd /content/drive/MyDrive/pubmed_llm

# Optional secrets from Colab Secrets. It is OK if one is missing.
for name in ['HF_TOKEN', 'GOOGLE_SERVICE_ACCOUNT_JSON', 'GOOGLE_DRIVE_DB_FILE_ID', 'ENTREZ_EMAIL']:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            print(f'Loaded secret: {name}')
    except Exception:
        pass

def run(command):
    print('\n$ ' + command, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}')

run('pip install -q -r requirements-worker.txt')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(os.environ['HF_TOKEN'])

print('Setup complete.')

## 2. Settings

Most lab members only need to adjust the numbers below.

For normal weekly/monthly maintenance, leave `REFRESH_BEFORE` blank. To finish a paused monthly campaign, set it to the campaign start date, for example `2026-06-08`.

In [ ]:
DB_PATH = '/content/drive/MyDrive/pubmed_llm/gene_function_lab/gene_function_lab.db'
CACHE_DIR = '/content/drive/MyDrive/pubmed_llm/functional_study_cache'

# Queue work from the website.
MAX_QUEUE_REQUESTS = 5
MAX_PAPERS = 300
RETRY_ERRORS = True
RESET_INTERRUPTED = True

# Existing-gene refresh.
REFRESH_EXISTING_GENES = True
UPDATE_INTERVAL_DAYS = 30
REFRESH_BEFORE = ''      # Example: '2026-06-08' to finish a paused monthly run.
MAX_REFRESH_GENES = 15
REFRESH_MAX_PAPERS = 300

# Upload DB to Drive after the batch finishes.
UPLOAD_AT_END = True

## 3. Check Status

Run before and after maintenance. It shows queue state and a preview of genes that still need refresh.

In [ ]:
cmd = (
    f'python -u scripts/check_queue_status.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--limit 30 '
)
if REFRESH_BEFORE:
    cmd += f'--refresh-before {shlex.quote(REFRESH_BEFORE)} '
else:
    cmd += f'--stale-days {UPDATE_INTERVAL_DAYS} '
run(cmd)

## 4. Main Maintenance Run

This is the main cell. It processes queue rows first, then refreshes a bounded number of stale existing genes.

If the run stops because Colab disconnects, run **Check Status**, then run this cell again.

In [ ]:
cmd = (
    f'python -u scripts/process_queue.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--cache-dir {shlex.quote(CACHE_DIR)} '
    f'--max-requests {MAX_QUEUE_REQUESTS} '
    f'--max-papers {MAX_PAPERS} '
)
if RETRY_ERRORS:
    cmd += '--retry-errors '
if RESET_INTERRUPTED:
    cmd += '--reset-processing '
if REFRESH_EXISTING_GENES:
    cmd += (
        f'--refresh-stale '
        f'--update-interval-days {UPDATE_INTERVAL_DAYS} '
        f'--max-refresh-genes {MAX_REFRESH_GENES} '
        f'--refresh-max-papers {REFRESH_MAX_PAPERS} '
    )
    if REFRESH_BEFORE:
        cmd += f'--refresh-before {shlex.quote(REFRESH_BEFORE)} '
if UPLOAD_AT_END:
    cmd += '--upload-at-end '
run(cmd)

## 5. Final Check

Run after the main cell. If `Queue pending`, `Queue processing`, `Queue error`, and `Genes needing refresh` are all acceptable, the maintenance run is done.

In [ ]:
cmd = (
    f'python -u scripts/check_queue_status.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--limit 30 '
)
if REFRESH_BEFORE:
    cmd += f'--refresh-before {shlex.quote(REFRESH_BEFORE)} '
else:
    cmd += f'--stale-days {UPDATE_INTERVAL_DAYS} '
run(cmd)